# 🟢 Google Data Engineer – Page 1 (Q1-Q411)
**Source:** Google Coding Questions – DataInterview.com (Page 1)

Questions covered:
| # | Title | Category | Difficulty |
|---|-------|----------|-----------|
| Q41 | User Reaction Rate | SQL | MEDIUM |
| Q55 | CTR by Website Type | SQL | HARD |
| Q65 | Insert Interval | Python | MEDIUM |
| Q80 | Sliding Window Maximum | Python | HARD |
| Q96 | Counts of Email Labels | SQL | MEDIUM |
| Q127 | Best Actors per Genre | SQL | HARD |
| Q150 | House Robber | Python | MEDIUM |


---
## Q41 · User Reaction Rate (Database – MEDIUM)

### Problem
Given a table of user reactions (likes, dislikes, comments) on posts, compute the **reaction rate**
(total reactions / total posts viewed) grouped by user.

**Table: reactions**
| user_id | post_id | action | action_date |
|---------|---------|--------|------------|
| 1       | 101     | like   | 2023-01-01 |
| 1       | 102     | view   | 2023-01-01 |
| 2       | 101     | view   | 2023-01-02 |

> **Reaction Rate** = (likes + dislikes + comments) / total views per user

---
### 🧠 How to Remember (Step by Step)
```
1. THINK: "Rate = reactions ÷ views" → need to COUNT two things separately
2. FILTER: use CASE WHEN or WHERE to split reactions vs views
3. GROUP BY user_id
4. DIVIDE: reaction_count / view_count
5. ROUND to 2 decimal places
```


In [ ]:
import sqlite3
import pandas as pd

# Setup
conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE reactions (
    user_id INT, post_id INT, action TEXT, action_date TEXT
)
""")
conn.executemany("INSERT INTO reactions VALUES (?,?,?,?)", [
    (1, 101, 'like',    '2023-01-01'),
    (1, 101, 'view',    '2023-01-01'),
    (1, 102, 'view',    '2023-01-02'),
    (1, 103, 'comment', '2023-01-03'),
    (2, 101, 'view',    '2023-01-01'),
    (2, 102, 'like',    '2023-01-02'),
    (2, 103, 'view',    '2023-01-03'),
    (3, 101, 'view',    '2023-01-01'),
])

print("Sample data:")
print(pd.read_sql("SELECT * FROM reactions", conn))


### ✅ Solution 1 – CASE WHEN in SELECT (Brute Force)
**Time:** O(n) | **Space:** O(u) where u = distinct users


In [ ]:
# Solution 1: CASE WHEN inline aggregation
sql1 = """
SELECT
    user_id,
    ROUND(
        1.0 * SUM(CASE WHEN action IN ('like','dislike','comment') THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN action = 'view' THEN 1 ELSE 0 END), 0),
        2
    ) AS reaction_rate
FROM reactions
GROUP BY user_id
ORDER BY user_id
"""
print("Solution 1 - CASE WHEN:")
print(pd.read_sql(sql1, conn))


### ✅ Solution 2 – Subquery with CTEs
**Time:** O(n) | **Space:** O(u)


In [ ]:
# Solution 2: CTE approach - cleaner separation of concerns
sql2 = """
WITH views AS (
    SELECT user_id, COUNT(*) AS total_views
    FROM reactions WHERE action = 'view'
    GROUP BY user_id
),
reacts AS (
    SELECT user_id, COUNT(*) AS total_reactions
    FROM reactions WHERE action IN ('like','dislike','comment')
    GROUP BY user_id
)
SELECT v.user_id,
       ROUND(1.0 * COALESCE(r.total_reactions, 0) / v.total_views, 2) AS reaction_rate
FROM views v
LEFT JOIN reacts r ON v.user_id = r.user_id
ORDER BY v.user_id
"""
print("Solution 2 - CTE:")
print(pd.read_sql(sql2, conn))


### ✅ Solution 3 – Window Function + Filter (Best for analytics)
**Time:** O(n log n) | **Space:** O(n)


In [ ]:
# Solution 3: Window functions - most readable for OLAP workloads
sql3 = """
SELECT DISTINCT user_id,
    ROUND(
        1.0 * SUM(CASE WHEN action != 'view' THEN 1 ELSE 0 END) OVER (PARTITION BY user_id)
        / NULLIF(SUM(CASE WHEN action = 'view' THEN 1 ELSE 0 END) OVER (PARTITION BY user_id), 0),
        2
    ) AS reaction_rate
FROM reactions
ORDER BY user_id
"""
print("Solution 3 - Window Functions:")
print(pd.read_sql(sql3, conn))


---
## Q55 · CTR by Website Type (Database – HARD)

### Problem
Given `events` (user, event_type, website_type, ts) where event_type is 'click' or 'impression',
compute **CTR = clicks / impressions** by website_type.

**CTR (Click-Through Rate)** = total clicks / total impressions

---
### 🧠 How to Remember (Step by Step)
```
1. CTR = clicks / impressions  (memorize this formula!)
2. JOIN or CASE WHEN to separate clicks vs impressions
3. GROUP BY website_type
4. Handle division by zero with NULLIF
5. Multiply by 100 if percent needed
```


In [ ]:
conn.execute("""
CREATE TABLE events (
    user_id INT, event_type TEXT, website_type TEXT, ts TEXT
)
""")
conn.executemany("INSERT INTO events VALUES (?,?,?,?)", [
    (1, 'impression', 'news',   '2023-01-01'),
    (1, 'click',      'news',   '2023-01-01'),
    (2, 'impression', 'news',   '2023-01-02'),
    (2, 'impression', 'sports', '2023-01-02'),
    (2, 'click',      'sports', '2023-01-02'),
    (3, 'impression', 'news',   '2023-01-03'),
    (3, 'impression', 'sports', '2023-01-03'),
    (4, 'impression', 'news',   '2023-01-03'),
    (4, 'click',      'news',   '2023-01-03'),
])
print("Events data:")
print(pd.read_sql("SELECT * FROM events", conn))


In [ ]:
# Solution 1: CASE WHEN - most common interview answer
sql_ctr1 = """
SELECT website_type,
       COUNT(CASE WHEN event_type = 'click' THEN 1 END) AS clicks,
       COUNT(CASE WHEN event_type = 'impression' THEN 1 END) AS impressions,
       ROUND(
           100.0 * COUNT(CASE WHEN event_type = 'click' THEN 1 END)
           / NULLIF(COUNT(CASE WHEN event_type = 'impression' THEN 1 END), 0),
           2
       ) AS ctr_pct
FROM events
GROUP BY website_type
ORDER BY ctr_pct DESC
"""
print("CTR Solution 1 - CASE WHEN:")
print(pd.read_sql(sql_ctr1, conn))


In [ ]:
# Solution 2: Self-JOIN approach
sql_ctr2 = """
WITH clicks AS (
    SELECT website_type, COUNT(*) AS click_count
    FROM events WHERE event_type = 'click'
    GROUP BY website_type
),
impressions AS (
    SELECT website_type, COUNT(*) AS imp_count
    FROM events WHERE event_type = 'impression'
    GROUP BY website_type
)
SELECT i.website_type,
       COALESCE(c.click_count, 0) AS clicks,
       i.imp_count AS impressions,
       ROUND(100.0 * COALESCE(c.click_count, 0) / i.imp_count, 2) AS ctr_pct
FROM impressions i
LEFT JOIN clicks c ON i.website_type = c.website_type
ORDER BY ctr_pct DESC
"""
print("CTR Solution 2 - CTE + LEFT JOIN:")
print(pd.read_sql(sql_ctr2, conn))


In [ ]:
# Solution 3: AVG trick - elegant one-liner
# AVG(event_type = 'click') works in MySQL; adapt for SQLite
sql_ctr3 = """
SELECT website_type,
       SUM(CASE WHEN event_type='click' THEN 1.0 ELSE 0 END) /
       SUM(CASE WHEN event_type='impression' THEN 1.0 ELSE 0 END) AS ctr,
       COUNT(*) AS total_events
FROM events
GROUP BY website_type
"""
print("CTR Solution 3 - Ratio trick:")
print(pd.read_sql(sql_ctr3, conn))


---
## Q65 · Insert Interval (Data Structures – MEDIUM)

### Problem
Given a sorted list of non-overlapping intervals and a new interval, insert it and merge overlaps.

**Example:**
- intervals = [[1,3],[6,9]], newInterval = [2,5]
- Output: [[1,5],[6,9]]

---
### 🧠 How to Remember (Step by Step)
```
1. THREE CASES for each existing interval vs new interval:
   a) Existing ends BEFORE new starts → add existing (no overlap)
   b) Existing starts AFTER new ends → add new, then rest
   c) OVERLAP → merge: new = [min(starts), max(ends)]
2. Don't forget to add the merged interval at the end!
3. Key check: interval[1] < new[0]  → before
              interval[0] > new[1]  → after
              else                  → merge
```


In [ ]:
from typing import List

# Solution 1: Brute Force - add all, sort, merge
# Time: O(n log n)  Space: O(n)
def insert_interval_brute(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    """Add new interval, sort all, then merge overlaps."""
    all_intervals = intervals + [new]
    all_intervals.sort(key=lambda x: x[0])

    merged = [all_intervals[0]]
    for start, end in all_intervals[1:]:
        if start <= merged[-1][1]:          # overlap
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

# Test
print("Solution 1 (Brute Force):")
print(insert_interval_brute([[1,3],[6,9]], [2,5]))   # [[1,5],[6,9]]
print(insert_interval_brute([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]))  # [[1,2],[3,10],[12,16]]


In [ ]:
# Solution 2: Linear scan - Three-case logic
# Time: O(n)  Space: O(n)
def insert_interval_linear(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    """Single pass with 3 cases: before, overlap, after."""
    result = []
    i = 0
    n = len(intervals)

    # Case a: intervals that end before new starts (no overlap)
    while i < n and intervals[i][1] < new[0]:
        result.append(intervals[i])
        i += 1

    # Case b: merge overlapping intervals
    while i < n and intervals[i][0] <= new[1]:
        new[0] = min(new[0], intervals[i][0])
        new[1] = max(new[1], intervals[i][1])
        i += 1
    result.append(new)

    # Case c: intervals that start after new ends
    while i < n:
        result.append(intervals[i])
        i += 1

    return result

print("Solution 2 (Linear Scan):")
print(insert_interval_linear([[1,3],[6,9]], [2,5]))
print(insert_interval_linear([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]))


In [ ]:
# Solution 3: Pythonic one-liner with list comprehension
# Time: O(n)  Space: O(n)
def insert_interval_pythonic(intervals: List[List[int]], new: List[int]) -> List[List[int]]:
    """Elegant Python using conditional building."""
    s, e = new
    left  = [i for i in intervals if i[1] < s]       # completely before
    right = [i for i in intervals if i[0] > e]       # completely after
    # overlapping: merge all into one
    overlap = [i for i in intervals if i[0] <= e and i[1] >= s]
    if overlap:
        s = min(s, overlap[0][0])
        e = max(e, overlap[-1][1])
    return left + [[s, e]] + right

print("Solution 3 (Pythonic):")
print(insert_interval_pythonic([[1,3],[6,9]], [2,5]))
print(insert_interval_pythonic([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]))

print("\n⏱ Complexity Summary:")
print("| Solution | Time     | Space |")
print("|----------|----------|-------|")
print("| Brute    | O(n logn)| O(n)  |")
print("| Linear   | O(n)     | O(n)  |")
print("| Pythonic | O(n)     | O(n)  |")


---
## Q80 · Sliding Window Maximum (Algorithms – HARD)

### Problem
Given an array and window size k, return the maximum value in each sliding window.

**Example:** nums=[1,3,-1,-3,5,3,6,7], k=3 → [3,3,5,5,6,7]

---
### 🧠 How to Remember (Step by Step)
```
1. BRUTE: nested loop O(nk) - too slow
2. DEQUE trick: store INDICES (not values), keep decreasing order
   - Remove from FRONT if out of window (index <= i - k)
   - Remove from BACK while back element <= current (useless)
   - Front of deque is always the MAXIMUM
3. Key insight: deque is MONOTONICALLY DECREASING → front = max
```


In [ ]:
from collections import deque

# Solution 1: Brute Force
# Time: O(n*k)  Space: O(1)
def max_sliding_window_brute(nums: List[int], k: int) -> List[int]:
    """Check every window of size k."""
    if not nums or k == 0:
        return []
    return [max(nums[i:i+k]) for i in range(len(nums) - k + 1)]

nums = [1,3,-1,-3,5,3,6,7]
print("Solution 1 (Brute Force):", max_sliding_window_brute(nums, 3))


In [ ]:
# Solution 2: Max-Heap (Priority Queue)
# Time: O(n log k)  Space: O(k)
import heapq

def max_sliding_window_heap(nums: List[int], k: int) -> List[int]:
    """Use max heap; lazily remove out-of-window elements."""
    heap = []  # (-val, index)
    result = []

    for i, num in enumerate(nums):
        heapq.heappush(heap, (-num, i))
        # remove elements outside window
        while heap[0][1] <= i - k:
            heapq.heappop(heap)
        if i >= k - 1:
            result.append(-heap[0][0])
    return result

print("Solution 2 (Heap):", max_sliding_window_heap(nums, 3))


In [ ]:
# Solution 3: Monotonic Deque ← OPTIMAL for interviews
# Time: O(n)  Space: O(k)
def max_sliding_window_deque(nums: List[int], k: int) -> List[int]:
    """
    Monotonic decreasing deque stores indices.
    Front = max of current window.
    """
    dq = deque()   # stores indices, front = max
    result = []

    for i, num in enumerate(nums):
        # 1) Remove indices outside window from front
        if dq and dq[0] <= i - k:
            dq.popleft()

        # 2) Remove smaller elements from back (they're useless)
        while dq and nums[dq[-1]] <= num:
            dq.pop()

        dq.append(i)

        # 3) Window is full → record max (front of deque)
        if i >= k - 1:
            result.append(nums[dq[0]])

    return result

print("Solution 3 (Monotonic Deque):", max_sliding_window_deque(nums, 3))
print("\n⏱ Complexity Summary:")
print("| Solution   | Time      | Space |")
print("|------------|-----------|-------|")
print("| Brute      | O(n*k)    | O(1)  |")
print("| Heap       | O(n log k)| O(k)  |")
print("| Deque ✅   | O(n)      | O(k)  |")


---
## Q96 · Counts of Email Labels (Database – MEDIUM)

### Problem
Given a Gmail `emails` table with columns (id, from_user, to_user, label),
count emails per label. Labels include: 'Promotions', 'Social', 'Updates', etc.

---
### 🧠 How to Remember
```
1. Simple GROUP BY + COUNT(*)
2. To show labels with 0 emails → LEFT JOIN with label list
3. For percentage → COUNT per group / total * 100
```


In [ ]:
conn.execute("""
CREATE TABLE IF NOT EXISTS emails (
    id INT, from_user TEXT, to_user TEXT, label TEXT
)
""")
conn.executemany("INSERT INTO emails VALUES (?,?,?,?)", [
    (1, 'a@g.com', 'b@g.com', 'Promotions'),
    (2, 'c@g.com', 'b@g.com', 'Social'),
    (3, 'd@g.com', 'b@g.com', 'Promotions'),
    (4, 'e@g.com', 'b@g.com', 'Updates'),
    (5, 'f@g.com', 'b@g.com', 'Social'),
    (6, 'g@g.com', 'b@g.com', 'Promotions'),
    (7, 'h@g.com', 'b@g.com', 'Primary'),
])

# Solution 1: Simple GROUP BY COUNT
sql_email1 = """
SELECT label, COUNT(*) AS email_count
FROM emails
GROUP BY label
ORDER BY email_count DESC
"""
print("Solution 1 - GROUP BY:")
print(pd.read_sql(sql_email1, conn))


In [ ]:
# Solution 2: WITH percentage
sql_email2 = """
SELECT label,
       COUNT(*) AS email_count,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM emails
GROUP BY label
ORDER BY email_count DESC
"""
print("Solution 2 - With percentage (window function):")
print(pd.read_sql(sql_email2, conn))


In [ ]:
# Solution 3: Ranked labels
sql_email3 = """
WITH counts AS (
    SELECT label, COUNT(*) AS email_count
    FROM emails
    GROUP BY label
)
SELECT label, email_count,
       RANK() OVER (ORDER BY email_count DESC) AS rank
FROM counts
"""
print("Solution 3 - Ranked by count:")
print(pd.read_sql(sql_email3, conn))


---
## Q127 · Best Actors per Genre (Database – HARD)

### Problem
Given `movies` (id, title, genre, rating) and `actors` (movie_id, actor_name),
find the top-rated actor per genre (actor with highest avg rating in that genre).

---
### 🧠 How to Remember
```
1. JOIN movies + actors on movie_id
2. GROUP BY genre, actor_name → get avg rating per actor per genre
3. RANK() OVER (PARTITION BY genre ORDER BY avg_rating DESC)
4. Filter WHERE rank = 1
```


In [ ]:
conn.execute("CREATE TABLE IF NOT EXISTS movies (id INT, title TEXT, genre TEXT, rating REAL)")
conn.execute("CREATE TABLE IF NOT EXISTS actors (movie_id INT, actor_name TEXT)")
conn.executemany("INSERT INTO movies VALUES (?,?,?,?)", [
    (1, 'Movie A', 'Action',  8.5),
    (2, 'Movie B', 'Action',  7.0),
    (3, 'Movie C', 'Comedy',  9.0),
    (4, 'Movie D', 'Comedy',  6.5),
    (5, 'Movie E', 'Drama',   8.0),
    (6, 'Movie F', 'Drama',   9.5),
])
conn.executemany("INSERT INTO actors VALUES (?,?)", [
    (1, 'Tom H'), (2, 'Tom H'), (3, 'Amy A'),
    (4, 'Ben B'), (5, 'Tom H'), (6, 'Amy A'),
    (1, 'Amy A'),
])

# Solution 1: Subquery + GROUP BY
sql_actors1 = """
SELECT m.genre, a.actor_name, ROUND(AVG(m.rating),2) AS avg_rating
FROM movies m
JOIN actors a ON m.id = a.movie_id
GROUP BY m.genre, a.actor_name
ORDER BY m.genre, avg_rating DESC
"""
print("Solution 1 - All actors per genre (ranked):")
print(pd.read_sql(sql_actors1, conn))


In [ ]:
# Solution 2: RANK() to pick top-1 per genre
sql_actors2 = """
WITH actor_ratings AS (
    SELECT m.genre, a.actor_name, AVG(m.rating) AS avg_rating
    FROM movies m
    JOIN actors a ON m.id = a.movie_id
    GROUP BY m.genre, a.actor_name
),
ranked AS (
    SELECT *, RANK() OVER (PARTITION BY genre ORDER BY avg_rating DESC) AS rnk
    FROM actor_ratings
)
SELECT genre, actor_name, ROUND(avg_rating, 2) AS avg_rating
FROM ranked WHERE rnk = 1
ORDER BY genre
"""
print("Solution 2 - Top actor per genre (RANK):")
print(pd.read_sql(sql_actors2, conn))


In [ ]:
# Solution 3: ROW_NUMBER to handle ties differently
sql_actors3 = """
WITH actor_ratings AS (
    SELECT m.genre, a.actor_name,
           ROUND(AVG(m.rating), 2) AS avg_rating,
           COUNT(DISTINCT m.id) AS movie_count
    FROM movies m JOIN actors a ON m.id = a.movie_id
    GROUP BY m.genre, a.actor_name
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY genre ORDER BY avg_rating DESC, movie_count DESC) AS rn
    FROM actor_ratings
)
SELECT genre, actor_name, avg_rating, movie_count
FROM ranked WHERE rn = 1
"""
print("Solution 3 - ROW_NUMBER with tie-breaking by movie count:")
print(pd.read_sql(sql_actors3, conn))


---
## Q150 · House Robber (Algorithms – MEDIUM)

### Problem
Given amounts in houses in a row, find max money you can rob without robbing adjacent houses.

**Example:** [2,7,9,3,1] → 12 (rob houses 0,2,4 → 2+9+1=12)

---
### 🧠 How to Remember (Step by Step)
```
MANTRA: "At each house, I have TWO choices: rob it or skip it"
  - Rob house i:  dp[i] = dp[i-2] + nums[i]
  - Skip house i: dp[i] = dp[i-1]
  - Take MAX of both

Space optimize: only need prev2 and prev1 (rolling variables)
```


In [ ]:
# Solution 1: Recursion with memoization (Top-down DP)
# Time: O(n)  Space: O(n)
from functools import lru_cache

def house_robber_memo(nums: List[int]) -> int:
    """Top-down: rob(i) = max(rob(i-2)+nums[i], rob(i-1))"""
    @lru_cache(maxsize=None)
    def rob(i):
        if i < 0: return 0
        return max(rob(i-2) + nums[i], rob(i-1))
    return rob(len(nums) - 1)

print("Solution 1 (Memoization):", house_robber_memo([2,7,9,3,1]))  # 12
print("Solution 1:", house_robber_memo([1,2,3,1]))  # 4


In [ ]:
# Solution 2: Bottom-up DP array
# Time: O(n)  Space: O(n)
def house_robber_dp(nums: List[int]) -> int:
    """Build dp array: dp[i] = max loot up to house i."""
    if not nums: return 0
    n = len(nums)
    if n == 1: return nums[0]

    dp = [0] * n
    dp[0] = nums[0]
    dp[1] = max(nums[0], nums[1])

    for i in range(2, n):
        dp[i] = max(dp[i-2] + nums[i], dp[i-1])

    return dp[-1]

print("Solution 2 (DP Array):", house_robber_dp([2,7,9,3,1]))  # 12


In [ ]:
# Solution 3: Space-optimized DP (rolling 2 vars) ← OPTIMAL
# Time: O(n)  Space: O(1)
def house_robber_optimized(nums: List[int]) -> int:
    """Only keep prev2 and prev1 - no array needed."""
    prev2, prev1 = 0, 0
    for num in nums:
        curr = max(prev2 + num, prev1)  # rob or skip
        prev2, prev1 = prev1, curr
    return prev1

print("Solution 3 (O(1) Space):", house_robber_optimized([2,7,9,3,1]))  # 12
print("Solution 3:", house_robber_optimized([1,2,3,1]))  # 4

print("\n⏱ Complexity Summary:")
print("| Solution       | Time | Space |")
print("|----------------|------|-------|")
print("| Memoization    | O(n) | O(n)  |")
print("| DP Array       | O(n) | O(n)  |")
print("| Rolling Vars ✅| O(n) | O(1)  |")
